In [ ]:
import os
import pandas as pd
import re


try:
    from openpyxl import load_workbook
except ImportError:
    print("Public message removed for release.")
    exit()

In [ ]:

def get_filename_without_extension(xlsx_path):
    
    
    base_name = os.path.basename(xlsx_path)
    
    file_name_without_extension = os.path.splitext(base_name)[0]
    return file_name_without_extension

In [ ]:

def extract_energy_from_out(energy_out_file_path):
    pattern = r'^FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)' 
    with open(energy_out_file_path, 'r') as file:
        for line in file:
            match = re.match(pattern, line)
            if match:
                return float(match.group(1))
    return None

In [ ]:
import re

def extract_homo_lumo(energy_out_file_path):
    
    
    with open(energy_out_file_path, 'r') as file:
        lines = file.readlines()
    
    
    start_index = None
    for idx in range(len(lines) - 1, -1, -1):
        if "ORBITAL ENERGIES" in lines[idx]:
            start_index = idx
            break
    
    if start_index is None:
        start_index = 0

    
    
    
    pattern = re.compile(r'^\s*(\d+)\s+([\d\.]+)\s+(-?[\d\.]+)\s+(-?[\d\.]+)')
    
    header_found = False  
    prev_occ = None       
    prev_eV = None        

    
    for line in lines[start_index:]:
        
        if not header_found:
            if "NO" in line and "OCC" in line and "E(Eh)" in line and "E(eV)" in line:
                header_found = True  
            continue  
        
        
        match = pattern.match(line)
        if match:
            
            
            line_no = int(match.group(1))
            occ = float(match.group(2))
            energy_eh = float(match.group(3))  
            energy_ev = float(match.group(4))
            
            
            if prev_occ is None:
                prev_occ = occ
                prev_eV = energy_ev
            else:
                
                if prev_occ != 0.0 and occ == 0.0:
                    homo = prev_eV  
                    lumo = energy_ev 
                    return homo, lumo
                
                prev_occ = occ
                prev_eV = energy_ev

    
    raise ValueError("Public message removed for release.")

In [ ]:
def extract_dipole_moment(opt_out_file_path):
    
    
    
    
    
    
    
    
    
    pattern = re.compile(r"^Magnitude \(Debye\)\s*:\s*([\d\.\-eE]+)")
    
    
    with open(opt_out_file_path, 'r') as file:
        
        for line in file:
            
            match = pattern.match(line)
            if match:
                
                dipole_str = match.group(1)
                try:
                    
                    dipole_value = float(dipole_str)
                    return dipole_value
                except ValueError:
                    
                    return None
    
    return None

In [ ]:

def extract_gibbs_correction(opt_out_file_path):
    
    
    
    
    
    
    
    
    pattern = re.compile(r"G-E\(el\)\s*\.*\s*([\d\.\-]+)\s*Eh", re.IGNORECASE)
    
    
    with open(opt_out_file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                try:
                    gibbs_correction = float(match.group(1))
                    return gibbs_correction
                except ValueError:
                    return None
    
    return None

In [ ]:

def extract_inner_energy_correction(opt_out_file_path):
    
    
    
    
    
    
    
    
    
    
    
    
    
    pattern = re.compile(
        r"^(Zero point energy|Thermal vibrational correction|Thermal rotational correction|Thermal translational correction)\s*\.*\s*([\d\.\-]+)\s*Eh",
        re.IGNORECASE | re.MULTILINE
    )
    
    total_correction = 0.0  
    found = False         
    
    
    with open(opt_out_file_path, 'r') as file:
        content = file.read()
        
        matches = pattern.findall(content)
        
        
        for label, value_str in matches:
            try:
                total_correction += float(value_str)
                found = True
            except ValueError:
                
                continue

    
    return total_correction if found else None

In [ ]:


def extract_enthalpy_correction(opt_out_file_path):
    
    
    
    
    
    
    pattern = re.compile(r"Thermal Enthalpy correction\s*\.*\s*([\d\.\-]+)\s*Eh", re.IGNORECASE)
    
    
    with open(opt_out_file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                try:
                    enthalpy_correction = float(match.group(1))
                    return enthalpy_correction
                except ValueError:
                    return None
    
    return None

In [ ]:


def extract_entropy(opt_out_file_path):
    
    
    
    
    
    
    
    
    pattern = re.compile(r"Final entropy term\s*\.*\s*([\d\.\-]+)\s*Eh", re.IGNORECASE)
    
    with open(opt_out_file_path, 'r') as file:
        for line in file:
            match = pattern.search(line)
            if match:
                try:
                    entropy = float(match.group(1))
                    return entropy
                except ValueError:
                    return None
    return None

In [ ]:


def extract_properties_from_energycal(excel_file, energy_success_dir):
    
    df = pd.read_excel(excel_file)

    df['Component A Energy (Hatree)'] = 0.0 
    df['Component A HOMO (eV)'] = 0.0 
    df['Component A LUMO (eV)'] = 0.0 
    
    df['Component B Energy (Hatree)'] = 0.0 
    df['Component B HOMO (eV)'] = 0.0 
    df['Component B LUMO (eV)'] = 0.0 
    
    
    for filename in os.listdir(energy_success_dir):
        if filename.endswith('.out'):
            
            component_name = filename.rsplit('.', 1)[0]
            
            energy = extract_energy_from_out(os.path.join(energy_success_dir, filename)) 
            homo, lumo = extract_homo_lumo(os.path.join(energy_success_dir, filename)) 

            
            energy = float(energy) if energy else 0.0
            homo = float(homo) if homo else 0.0
            lumo = float(lumo) if lumo else 0.0
            
            
            
            df.loc[df['Component Name A'] == component_name, 'Component A Energy (Hatree)'] = float(energy)
            df.loc[df['Component Name A'] == component_name, 'Component A HOMO (eV)'] = homo
            df.loc[df['Component Name A'] == component_name, 'Component A LUMO (eV)'] = lumo
            df.loc[df['Component Name B'] == component_name, 'Component B Energy (Hatree)'] = float(energy)
            df.loc[df['Component Name B'] == component_name, 'Component B HOMO (eV)'] = homo
            df.loc[df['Component Name B'] == component_name, 'Component B LUMO (eV)'] = lumo

    
    
    df.to_excel(excel_file, index=False)

In [ ]:

def extract_properties_from_opt_freq(excel_file, opt_success_dir):
    
    
    df = pd.read_excel(excel_file)
    
    df["Component A Inner energy correction (Hatree)"] = 0.0 
    df["Component A Thermal correction to Enthalpy (Hatree)"] = 0.0 
    df["Component A Thermal correction to Gibbs Free Energy (Hatree)"] = 0.0 
    df['Component A Entropy (Hatree)'] = 0.0 
    df['Component A Dipole (Debye)'] = 0.0 

    df["Component B Inner energy correction (Hatree)"] = 0.0 
    df["Component B Thermal correction to Enthalpy (Hatree)"] = 0.0 
    df["Component B Thermal correction to Gibbs Free Energy (Hatree)"] = 0.0 
    df['Component B Entropy (Hatree)'] = 0.0 
    df['Component B Dipole (Debye)'] = 0.0 
    
    
    for filename in os.listdir(opt_success_dir):
        if filename.endswith('.out'):
            
            component_name = filename.rsplit('.', 1)[0]
            
            inner_energy_correction = extract_inner_energy_correction(os.path.join(opt_success_dir, filename)) 
            gibbs_correction = extract_gibbs_correction(os.path.join(opt_success_dir, filename)) 
            enthalpy_correction = extract_enthalpy_correction(os.path.join(opt_success_dir, filename)) 
            entropy = extract_entropy(os.path.join(opt_success_dir, filename)) 
            dipole = extract_dipole_moment(os.path.join(opt_success_dir, filename)) 
            
            
            dipole = float(dipole) if dipole else 0.0
            
            df.loc[df['Component Name A'] == component_name, 'Component A Inner energy correction (Hatree)'] = inner_energy_correction
            df.loc[df['Component Name A'] == component_name, 'Component A Thermal correction to Gibbs Free Energy (Hatree)'] = gibbs_correction
            df.loc[df['Component Name A'] == component_name, 'Component A Thermal correction to Enthalpy (Hatree)'] = enthalpy_correction
            df.loc[df['Component Name A'] == component_name, 'Component A Entropy (Hatree)'] = entropy
            df.loc[df['Component Name A'] == component_name, 'Component A Dipole (Debye)'] = dipole

            df.loc[df['Component Name B'] == component_name, 'Component B Inner energy correction (Hatree)'] = inner_energy_correction
            df.loc[df['Component Name B'] == component_name, 'Component B Thermal correction to Gibbs Free Energy (Hatree)'] = gibbs_correction
            df.loc[df['Component Name B'] == component_name, 'Component B Thermal correction to Enthalpy (Hatree)'] = enthalpy_correction
            df.loc[df['Component Name B'] == component_name, 'Component B Entropy (Hatree)'] = entropy
            df.loc[df['Component Name B'] == component_name, 'Component B Dipole (Debye)'] = dipole
                   
    
    df.to_excel(excel_file, index=False)

In [ ]:

def data_processing(excel_file):
    
    df = pd.read_excel(excel_file)
       
    df["Component A Gibbs Free Energy (Hatree)"] = df['Component A Energy (Hatree)'] + df["Component A Thermal correction to Gibbs Free Energy (Hatree)"] 
    df["Component A Enthalpy (Hatree)"] = df['Component A Energy (Hatree)'] + df["Component A Inner energy correction (Hatree)"] + df["Component A Thermal correction to Enthalpy (Hatree)"] 
    df["Component A HOMO LUMO Gap (eV)"] = (df['Component A LUMO (eV)'] - df["Component A HOMO (eV)"]) 

    df["Component B Gibbs Free Energy (Hatree)"] = df['Component B Energy (Hatree)'] + df["Component B Thermal correction to Gibbs Free Energy (Hatree)"] 
    df["Component B Enthalpy (Hatree)"] = df['Component B Energy (Hatree)'] + df["Component B Inner energy correction (Hatree)"] + df["Component B Thermal correction to Enthalpy (Hatree)"] 
    df["Component B HOMO LUMO Gap (eV)"] = (df['Component B LUMO (eV)'] - df["Component B HOMO (eV)"]) 

    
    df.to_excel(excel_file, index=False)

In [ ]:

component_excel_path = 'Dimer.xlsx'
opt_success_dir = "component_gas/opt+freq/success"
energy_success_dir = "component_gas/energy/success"


extract_properties_from_energycal(component_excel_path, energy_success_dir)
extract_properties_from_opt_freq(component_excel_path, opt_success_dir)
data_processing(component_excel_path)

In [ ]:
df = pd.read_excel('Dimer.xlsx')
df